# CKF Track Reconstruction vs Truth Performance

Visualizes the performance of the ACTS Combinatorial Kalman Filter (CKF) track reconstruction
by comparing reconstructed track parameters against truth values.

This notebook serves as the baseline for evaluating ML-based track parameter regression
and for comparing performance across beam spot configurations.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent))
from data.datasets import TrackHitDataset, compute_truth_params

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Configuration

In [ ]:
# Define datasets to compare
SIM_BASE = '/global/cfs/cdirs/m4958/data/ColliderML/simulation'

datasets = {
    'Nominal (0,0,0)': f'{SIM_BASE}/hard_scatter/ttbar/v1/parquet',
    # Uncomment these once the shifted datasets are ready:
    # 'Shifted 25 um': f'{SIM_BASE}/beamspot_studies/ttbar_shifted_25um/v1/parquet',
    # 'Shifted 300 um': f'{SIM_BASE}/beamspot_studies/ttbar_shifted_300um/v1/parquet',
}

PARAM_NAMES = ['d0', 'z0', 'phi', 'theta', 'qop']
PARAM_UNITS = ['mm', 'mm', 'rad', 'rad', '1/GeV']
PARAM_LABELS = [
    r'$d_0$', r'$z_0$', r'$\phi$', r'$\theta$', r'$q/p$'
]

## 2. Load Data

In [ ]:
def load_reco_truth(parquet_base, n_files=1):
    """Load tracks and match to truth particles, returning reco and truth params."""
    ds = TrackHitDataset(parquet_base, max_hits=20, file_indices=list(range(n_files)))
    
    reco = np.stack([s['reco_params'] for s in ds.samples])
    truth = np.stack([s['truth_params'] for s in ds.samples])
    n_hits = np.array([s['n_hits'] for s in ds.samples])
    
    print(f'  Loaded {len(ds)} tracks from {n_files} file(s)')
    return reco, truth, n_hits

data = {}
for name, path in datasets.items():
    print(f'Loading {name}...')
    data[name] = load_reco_truth(path, n_files=1)

## 3. Residual Distributions

In [ ]:
def plot_residuals(data, param_idx, param_name, param_unit, param_label, nbins=100):
    """Plot residual (reco - truth) distributions for a single parameter."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: overlay residual histograms
    ax = axes[0]
    for name, (reco, truth, _) in data.items():
        residual = reco[:, param_idx] - truth[:, param_idx]
        # Clip to 3 sigma for better visualization
        sigma = np.std(residual)
        mean = np.mean(residual)
        mask = np.abs(residual - mean) < 5 * sigma
        ax.hist(residual[mask], bins=nbins, alpha=0.6, density=True,
                label=f'{name}\n$\\mu$={mean:.4g}, $\\sigma$={sigma:.4g}')
    ax.set_xlabel(f'{param_label} residual (reco - truth) [{param_unit}]')
    ax.set_ylabel('Density')
    ax.set_title(f'{param_label} Residual Distribution')
    ax.legend(fontsize=10)
    
    # Right: reco vs truth scatter
    ax = axes[1]
    for name, (reco, truth, _) in data.items():
        # Subsample for scatter plot
        n_plot = min(5000, len(reco))
        idx = np.random.choice(len(reco), n_plot, replace=False)
        ax.scatter(truth[idx, param_idx], reco[idx, param_idx], 
                   alpha=0.1, s=1, label=name)
    
    # Diagonal line
    lims = ax.get_xlim()
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect')
    ax.set_xlabel(f'{param_label} truth [{param_unit}]')
    ax.set_ylabel(f'{param_label} reco [{param_unit}]')
    ax.set_title(f'{param_label}: Reco vs Truth')
    ax.legend(fontsize=10)
    
    plt.tight_layout()
    return fig

for i, (name, unit, label) in enumerate(zip(PARAM_NAMES, PARAM_UNITS, PARAM_LABELS)):
    fig = plot_residuals(data, i, name, unit, label)
    plt.show()

## 4. Resolution vs eta

In [ ]:
def plot_resolution_vs_eta(data, param_idx, param_name, param_unit, param_label, n_bins=20):
    """Plot parameter resolution (std of residual) vs pseudorapidity."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for ds_name, (reco, truth, _) in data.items():
        # Compute eta from truth theta
        theta = truth[:, 3]  # truth theta
        eta = -np.log(np.tan(theta / 2))
        
        residual = reco[:, param_idx] - truth[:, param_idx]
        
        # Bin by eta
        eta_edges = np.linspace(-4, 4, n_bins + 1)
        eta_centers = 0.5 * (eta_edges[:-1] + eta_edges[1:])
        resolutions = []
        
        for j in range(n_bins):
            mask = (eta >= eta_edges[j]) & (eta < eta_edges[j+1])
            if mask.sum() > 10:
                resolutions.append(np.std(residual[mask]))
            else:
                resolutions.append(np.nan)
        
        ax.plot(eta_centers, resolutions, 'o-', label=ds_name, markersize=4)
    
    ax.set_xlabel(r'$\eta$')
    ax.set_ylabel(f'{param_label} resolution ($\\sigma$) [{param_unit}]')
    ax.set_title(f'{param_label} Resolution vs $\\eta$ (CKF)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

for i, (name, unit, label) in enumerate(zip(PARAM_NAMES, PARAM_UNITS, PARAM_LABELS)):
    fig = plot_resolution_vs_eta(data, i, name, unit, label)
    plt.show()

## 5. Resolution vs pT

In [ ]:
def plot_resolution_vs_pt(data, param_idx, param_name, param_unit, param_label, n_bins=20):
    """Plot parameter resolution vs transverse momentum."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for ds_name, (reco, truth, _) in data.items():
        # Compute pT from truth qop and theta
        qop = truth[:, 4]
        theta = truth[:, 3]
        p = np.abs(1.0 / (qop + 1e-10))
        pt = p * np.sin(theta)
        
        residual = reco[:, param_idx] - truth[:, param_idx]
        
        # Log-spaced pT bins
        pt_edges = np.logspace(np.log10(0.5), np.log10(200), n_bins + 1)
        pt_centers = np.sqrt(pt_edges[:-1] * pt_edges[1:])
        resolutions = []
        
        for j in range(n_bins):
            mask = (pt >= pt_edges[j]) & (pt < pt_edges[j+1])
            if mask.sum() > 10:
                resolutions.append(np.std(residual[mask]))
            else:
                resolutions.append(np.nan)
        
        ax.plot(pt_centers, resolutions, 'o-', label=ds_name, markersize=4)
    
    ax.set_xlabel(r'$p_T$ [GeV]')
    ax.set_ylabel(f'{param_label} resolution ($\\sigma$) [{param_unit}]')
    ax.set_title(f'{param_label} Resolution vs $p_T$ (CKF)')
    ax.set_xscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

for i, (name, unit, label) in enumerate(zip(PARAM_NAMES, PARAM_UNITS, PARAM_LABELS)):
    fig = plot_resolution_vs_pt(data, i, name, unit, label)
    plt.show()

## 6. Hit Count Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for ds_name, (_, _, n_hits) in data.items():
    ax.hist(n_hits, bins=np.arange(0.5, 25.5), alpha=0.6, density=True, label=ds_name)
ax.set_xlabel('Number of hits per track')
ax.set_ylabel('Fraction')
ax.set_title('Track Hit Multiplicity')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary Table

In [ ]:
print(f"{'Dataset':<25} {'Param':<8} {'Bias':>10} {'Resolution':>12} {'N tracks':>10}")
print('-' * 70)
for ds_name, (reco, truth, _) in data.items():
    for i, (name, unit) in enumerate(zip(PARAM_NAMES, PARAM_UNITS)):
        residual = reco[:, i] - truth[:, i]
        bias = np.mean(residual)
        resolution = np.std(residual)
        n = len(residual)
        print(f'{ds_name:<25} {name:<8} {bias:>10.4g} {resolution:>12.4g} {n:>10}')

## 8. Primary Vertex Position (Beam Spot Check)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ds_name, parquet_path in datasets.items():
    particles = pq.read_table(Path(parquet_path) / 'truth/particles' / 
                              sorted(Path(parquet_path).glob('truth/particles/*.parquet'))[0].name)
    
    # Get primary vertex positions (first primary particle per event)
    vx_list, vy_list, vz_list = [], [], []
    for evt_idx in range(min(len(particles), 1000)):
        primary = np.array(particles['primary'][evt_idx].as_py())
        vx = np.array(particles['vx'][evt_idx].as_py(), dtype=np.float32)
        vy = np.array(particles['vy'][evt_idx].as_py(), dtype=np.float32)
        vz = np.array(particles['vz'][evt_idx].as_py(), dtype=np.float32)
        
        prim_mask = primary == True
        if prim_mask.any():
            vx_list.append(vx[prim_mask][0])
            vy_list.append(vy[prim_mask][0])
            vz_list.append(vz[prim_mask][0])
    
    vx_arr = np.array(vx_list)
    vy_arr = np.array(vy_list)
    vz_arr = np.array(vz_list)
    
    axes[0].hist(vx_arr * 1000, bins=50, alpha=0.6, density=True,
                 label=f'{ds_name}\n$\\mu$={np.mean(vx_arr)*1000:.1f} um')
    axes[1].hist(vy_arr * 1000, bins=50, alpha=0.6, density=True,
                 label=f'{ds_name}\n$\\mu$={np.mean(vy_arr)*1000:.1f} um')
    axes[2].hist(vz_arr, bins=50, alpha=0.6, density=True,
                 label=f'{ds_name}\n$\\mu$={np.mean(vz_arr):.1f} mm')

axes[0].set_xlabel('Primary vertex $v_x$ [um]')
axes[1].set_xlabel('Primary vertex $v_y$ [um]')
axes[2].set_xlabel('Primary vertex $v_z$ [mm]')
for ax in axes:
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[0].set_title('Beam Spot X')
axes[1].set_title('Beam Spot Y')
axes[2].set_title('Beam Spot Z')
plt.suptitle('Primary Vertex Positions (Beam Spot)', fontsize=14)
plt.tight_layout()
plt.show()